# 03_training_and_experiments.ipynb

This notebook trains an initial XGBoost model using the processed dataset created during the feature engineering workflow.

The notebook covers dataset preparation, data splitting, baseline model training, model evaluation, and experiment tracking. The resulting model serves as a baseline for future hyperparameter tuning and model optimization.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [1]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and project modules.

In [2]:
from pathlib import Path

import boto3
import itertools
import json
import joblib

import mlflow
import mlflow.xgboost
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    EVALUATION_REPORT_KEY,
    MLFLOW_TRACKING_SERVER_ARN,
    MODEL_ARTIFACT_KEY,
    MODEL_ARTIFACT_NAME,
    TRAIN_DATA_KEY,
    VALIDATION_DATA_KEY,
    TEST_DATA_KEY,
    TARGET_COLUMN,
)

from src.train import train_baseline_model
from src.evaluate import evaluate_model

## Initialize AWS Clients

Initialize client for Amazon S3.

In [3]:
s3 = boto3.client("s3", region_name=AWS_REGION)

## Load Processed Dataset from Amazon S3

The processed dataset created during the feature engineering step is loaded from Amazon S3 for model training and evaluation.

In [5]:
train_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TRAIN_DATA_KEY,
)

train_df = pd.read_csv(
    train_obj["Body"]
)

validation_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=VALIDATION_DATA_KEY,
)

validation_df = pd.read_csv(
    validation_obj["Body"]
)

test_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TEST_DATA_KEY,
)

test_df = pd.read_csv(
    test_obj["Body"]
)

In [6]:
print(f"Train dataset shape: {train_df.shape}")
print(f"Validation dataset shape: {validation_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

## Inspect Processed Dataset

The processed dataset is inspected to verify its structure, feature representation, and overall readiness for model training.

This step confirms that the feature engineering workflow produced a clean dataset suitable for XGBoost.

In [7]:
train_df.head()

In [8]:
train_df.info()

In [9]:
train_df["class"].value_counts(normalize=True)

### Dataset Summary

The processed dataset contains engineered numerical features and a binary target variable.

No additional preprocessing is required before splitting the dataset for model training.

## Prepare Training, Validation, and Test Sets

Separate features and target variables for model training, validation, and final evaluation.

In [10]:
X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]

X_validation = validation_df.drop(columns=[TARGET_COLUMN])
y_validation = validation_df[TARGET_COLUMN]

X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

## Train Baseline XGBoost Model

A baseline XGBoost classifier is trained using the training dataset.

The baseline model provides an initial performance benchmark and serves as a reference for future experiment tracking and hyperparameter tuning.

In [11]:
model = train_baseline_model(X_train, y_train)

## Evaluate Baseline Model

The baseline model is evaluated on the validation dataset to assess its predictive performance.

Several classification metrics are calculated to establish a performance baseline before further model optimization.

In [12]:
train_accuracy = model.score(X_train, y_train)

print(f"Train Accuracy: {train_accuracy:.4f}")

In [17]:
results = evaluate_model(model, X_validation, y_validation)
validation_accuracy = results['accuracy']
validation_roc_auc = results['roc_auc']

print(f"Validation Accuracy: {validation_accuracy:.4f}")
print(f"Validation ROC AUC: {validation_roc_auc:.4f}")

print("\nClassification Report")
print(results["classification_report"])

### Baseline Model Results

The baseline XGBoost model achieved a validation accuracy of 66.7% and a ROC AUC score of 0.68.

The model shows signs of overfitting, with substantially higher performance on the training dataset than on the validation dataset.

These results serve as a baseline for future experiment tracking and hyperparameter tuning.

## Tracking Server

In [18]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_SERVER_ARN)
mlflow.set_experiment("aws-sagemaker-showcase")

## Track Baseline Experiment with MLflow

The baseline training run is logged to MLflow to capture model parameters, evaluation metrics, and model artifacts.

This enables experiment comparison, reproducibility, and model lifecycle tracking within SageMaker.

In [19]:
with mlflow.start_run(run_name="baseline-xgboost"):
    mlflow.log_params(
        {
            "n_estimators": 100,
            "max_depth": 6,
            "learning_rate": 0.1,
        }
    )

    mlflow.log_metric("validation_accuracy", validation_accuracy)
    mlflow.log_metric("validation_roc_auc", validation_roc_auc)

    mlflow.xgboost.log_model(
        model,
        name="xgboost_model",
    )

### Experiment Tracking Results

The baseline training run was successfully tracked using MLflow.

Model parameters, evaluation metrics, and model artifacts were stored in the SageMaker MLflow Tracking Server, enabling reproducibility and future comparison with additional training runs.

## Hyperparameter Tuning

A small grid search is used to evaluate multiple XGBoost configurations.

Each run is tracked in MLflow so that the validation performance of different parameter combinations can be compared directly.

In [21]:
param_grid = {
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [50, 100],
    "subsample": [0.8, 1.0],
}

param_combinations = list(
    itertools.product(
        param_grid["max_depth"],
        param_grid["learning_rate"],
        param_grid["n_estimators"],
        param_grid["subsample"],
    )
)

len(param_combinations)

In [23]:
best_auc = float("-inf")
best_params = None
best_model = None
best_run_name = None

for i, (max_depth, learning_rate, n_estimators, subsample) in enumerate(param_combinations, start=1):
    run_name = f"run_{i}"

    with mlflow.start_run(run_name=run_name):
        params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "random_state": 42,
        }

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_validation)
        y_proba = model.predict_proba(X_validation)[:, 1]

        accuracy = accuracy_score(y_validation, y_pred)
        precision = precision_score(y_validation, y_pred)
        recall = recall_score(y_validation, y_pred)
        f1 = f1_score(y_validation, y_pred)
        auc = roc_auc_score(y_validation, y_proba)

        mlflow.log_params(params)
        mlflow.log_metric("validation_accuracy", accuracy)
        mlflow.log_metric("validation_precision", precision)
        mlflow.log_metric("validation_recall", recall)
        mlflow.log_metric("validation_f1", f1)
        mlflow.log_metric("validation_roc_auc", auc)
        mlflow.xgboost.log_model(model, name=f"xgboost_model_{i}")

        print(
            f"{run_name}: "
            f"accuracy={accuracy:.4f}, precision={precision:.4f}, "
            f"recall={recall:.4f}, f1={f1:.4f}, auc={auc:.4f}"
        )

        if auc > best_auc:
            best_auc = auc
            best_params = params
            best_model = model
            best_run_name = run_name

In [25]:
print("Best run:", best_run_name)
print("Best params:", best_params)
print(f"Best validation ROC AUC: {best_auc:.4f}")

### Tuning Results

The grid search evaluated multiple XGBoost configurations and logged each run to MLflow.

The best model was selected based on validation ROC AUC, which is a suitable metric for credit risk classification.

## Final Model Evaluation

The best model identified during hyperparameter tuning is evaluated on the test dataset.

The test set was not used during model training or hyperparameter selection and therefore provides an unbiased estimate of the model's generalization performance.

In [26]:
results = evaluate_model(best_model, X_test, y_test)

print(f"Test Accuracy: {results['accuracy']:.4f}")
print(f"Test ROC AUC: {results['roc_auc']:.4f}")

print("\nClassification Report")
print(results["classification_report"])

In [27]:
final_results = {
    "best_run": best_run_name,
    "best_params": best_params,
    "validation_roc_auc": float(best_auc),
    "test_accuracy": float(results["accuracy"]),
    "test_roc_auc": float(results["roc_auc"]),
}

final_results

### Final Evaluation Results

The tuned XGBoost model achieved the best validation performance during hyperparameter tuning and was subsequently evaluated on the test dataset.

The resulting test metrics provide the final performance estimate and will be used in later stages of the machine learning lifecycle, including model registration, deployment, and pipeline automation.

## Upload Model Artifacts to Amazon S3

The final model artifact and evaluation results are uploaded to Amazon S3.

These artifacts will be used in subsequent stages of the machine learning lifecycle, including explainability analysis, model registration, deployment, and pipeline automation.

In [28]:
Path("data/evaluation").mkdir(
    parents=True,
    exist_ok=True,
)

local_evaluation_path = "data/evaluation/evaluation.json"

with open(local_evaluation_path, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Evaluation report uploaded to s3://{BUCKET_NAME}/{EVALUATION_REPORT_KEY}")

In [29]:
Path("data/models").mkdir(
    parents=True,
    exist_ok=True,
)

local_model_path = "data/models/best_model.pkl"

joblib.dump(
    best_model,
    local_model_path,
)

s3.upload_file(
    local_model_path,
    BUCKET_NAME,
    MODEL_ARTIFACT_KEY,
)

print(f"Model artifact uploaded to s3://{BUCKET_NAME}/{MODEL_ARTIFACT_KEY}")

### Result

The best performing XGBoost model and its evaluation results have been uploaded to Amazon S3.

These artifacts will be used in subsequent stages of the machine learning lifecycle, including explainability analysis, model registration, deployment, and pipeline automation.